# LLM4Rec: Mode A LoRA Ranking Finetune
Fair text-side counterpart to the Mode C ranking adapter.

**Prerequisites:** Run `Colab_Pipeline.ipynb` first so that `checkpoints/` artifacts exist.

**Setup:** Copy the `Finetuning/` folder and `scripts/finetune_lora_ranking.py` into your
Google Drive ECS172 folder. Then open this notebook in Colab and run all cells.

**Step order:**
1. Mount Drive & set path
2. Install missing dependency (unsloth already present from Pipeline)
3. Train Mode A LoRA (`llama31-1b-movielens-ranking-lora/`)
4. Evaluate Mode A (LoRA) on the 604 held-out test users
5. Evaluate Mode A (zero-shot) + Mode C (ranking adapter) side-by-side
6. Print results table

## Cell 1: Mount Google Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os
# IMPORTANT: Change this path if your ECS172 folder is somewhere else in Drive
PROJECT_DIR = '/content/drive/MyDrive/ECS172'
os.chdir(PROJECT_DIR)
print('Current directory:', os.getcwd())
!ls

# Verify the required artifacts from the pipeline are present
required = [
    'checkpoints/id_maps.json',
    'checkpoints/adapter_ranking.pt',
    'checkpoints/projected_embeddings_ranking.pt',
    'train.csv',
    'test_ranking_prompts.json',
    'test_ranking.csv',
    'scripts/finetune_lora_ranking.py',
]
for p in required:
    print(f"  {'OK     ' if os.path.exists(p) else 'MISSING'}  {p}")

Current directory: /content/drive/MyDrive/ECS172
checkpoints			   results_mode_A_zeroshot.json
Colab_AC_Eval.ipynb		   results_mode_C.json
Colab_Pipeline.ipynb		   results_mode_C_lora.json
explainability			   scripts
Finetuning			   src
huggingface_tokenizers_cache	   test_ranking.csv
llama31-1b-movielens-ranking-lora  test_ranking_prompts_AC.json
md_files			   test_ranking_prompts.json
README.md			   train.csv
requirements.txt		   train_ranking_prompts_5cand.json
results_ac.json			   unsloth_compiled_cache
results_mode_A.json		   val.csv
results_mode_A_lora.json	   val_prompts.json
  OK       checkpoints/id_maps.json
  OK       checkpoints/adapter_ranking.pt
  OK       checkpoints/projected_embeddings_ranking.pt
  OK       train.csv
  OK       test_ranking_prompts.json
  OK       test_ranking.csv
  OK       scripts/finetune_lora_ranking.py


## Cell 2: Install Dependencies
 We just make sure peft is present.

In [6]:
!pip install -q -r requirements.txt
!pip install -q peft
# Unsloth gives ~2x faster LLaMA loading/inference on Colab T4
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
print('Done.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 166.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 924.2/924.2 kB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 135.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 25.5 MB/s eta 0:00:00
Done.


## Cell 3: Train Mode A Ranking LoRA
Trains a LoRA on the exact same task as the eval: CE over the A-J letter logits at the
`Answer:` position. Same data, same seed as the ranking adapter — comparable training budget.

Validates each epoch by HR@1 (same as the ranking adapter) and saves the best checkpoint.

**Training time:** ~8 min on A100, ~20 min on L4, ~40 min on T4.

In [ ]:
!python scripts/finetune_lora_ranking.py \
    --model unsloth/Llama-3.2-1B-Instruct \
    --epochs 3 --max-train 5000 --val-fraction 0.1 \
    --lora-r 16 --lora-alpha 32 --lr 1e-4 --batch-size 8 \
    --output ./llama31-1b-movielens-ranking-lora

[device] cuda
[config] model=unsloth/Llama-3.2-1B-Instruct  epochs=3  max_train=5000  lr=0.0001
[data] 4500 train  /  500 val
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.10: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 146/146 [00:00<00:00, 239.29it/s]
config.json: 100% 894/894 [00:00<00:00, 4.62MB/s]
tokenizer_config.json: 100% 54.7k/54.7k [00

## Cell 4: Evaluate Mode A (LoRA) on the 604 Held-Out Test Users
Same scorer (`eval_ranking.py`), same test file, same metric as Mode C.

In [7]:
LLM_MODEL = './llama31-1b-movielens-ranking-lora'

!python scripts/eval_ranking.py --modes A \
    --model {LLM_MODEL} \
    --n-candidates 10

!cp results_mode_A.json results_mode_A_lora.json
print('\n[saved] Mode A (LoRA) results -> results_mode_A_lora.json')

[device] cuda
[data] 604 test examples
[setup] modes=['A'] candidates=10 chat_template=False
[llm] llama31-1b-movielens-ranking-lora
[load] loading LLM (first run may download base weights)...
[cache] projected_embeddings: projected_embeddings.pt
[load] PEFT adapter from llama31-1b-movielens-ranking-lora
[load] base model unsloth/Llama-3.2-1B-Instruct (download on first run ~2–5 min)
config.json: 100% 894/894 [00:00<00:00, 4.30MB/s]
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
model.safetensors: 100% 2.47G/2.47G [00:06<00:00, 361MB/s]
Loading weights: 100% 146/146 [00:00<00:00, 6119.89it/s]
generation_config.json: 100% 239/239 

## Cell 5: Evaluate Mode A (Zero-Shot) + Mode C (Ranking Adapter) Side-by-Side
Re-runs both conditions so all four are in the same output.

In [13]:
print('=== Mode A (zero-shot) + Mode C (ranking adapter) ===')
!python scripts/eval_ranking.py --modes A C \
    --model unsloth/Llama-3.2-1B-Instruct \
    --embedding-adapter  checkpoints/adapter_ranking.pt \
    --projected-embeddings checkpoints/projected_embeddings_ranking.pt \
    --n-candidates 10

!cp results_mode_A.json results_mode_A_zeroshot.json
!cp results_mode_C.json results_mode_C_base.json
print('\n[saved] Mode A (zero-shot) -> results_mode_A_zeroshot.json')
print('[saved] Mode C (base adapt) -> results_mode_C_base.json')

=== Mode A (zero-shot) + Mode C (ranking adapter) ===
[device] cuda
[data] 604 test examples
[setup] modes=['A', 'C'] candidates=10 chat_template=False
[llm] unsloth/Llama-3.2-1B-Instruct
[load] loading LLM (first run may download base weights)...
[adapter] adapter_ranking.pt  method=contrastive_ranking
[adapter] trained against: unsloth/Llama-3.2-1B-Instruct
[adapter] best_val_HR@1=0.4175
[cache] projected_embeddings: projected_embeddings_ranking.pt
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 146/146 [00:00<00:00, 6443.54it/s]
[load] projected_embeddings cache loaded (3510, 2048)
[coverage] Mode C: 288/6

## Cell 6: Results Table
Prints all four conditions: Random, Mode A (zero-shot), Mode A (LoRA), Mode C (ranking adapter).

In [9]:
import json
from pathlib import Path

# All three conditions with their result files (set up by Cells 4 & 5)
conditions = [
    ('Mode A (zero-shot)',     'results_mode_A_zeroshot.json'),
    ('Mode A (LoRA)',          'results_mode_A_lora.json'),
    ('Mode C (ranking adapt)', 'results_mode_C.json'),
]

# Determine K values from whichever file exists
ks, rand = [], {}
for _, f in conditions:
    p = Path(f)
    if p.exists():
        data = json.loads(p.read_text())
        ks   = sorted(int(k.split('@')[1]) for k in data['metrics'] if k.startswith('HR@'))
        rand = data.get('random_baseline', {})
        break

def fmt_row(label, m):
    hr   = ''.join(f"{m.get(f'HR@{k}',   0):>8.3f}" for k in ks)
    ndcg = ''.join(f"{m.get(f'NDCG@{k}', 0):>9.4f}" for k in ks)
    return f"{label:<30}{hr}{ndcg}"

hdr = f"{'Condition':<30}" + ''.join(f"{'HR@'+str(k):>8}" for k in ks) + ''.join(f"{'NDCG@'+str(k):>9}" for k in ks)
print(hdr)
print('-' * len(hdr))
if rand:
    print(fmt_row('Random', rand))
for label, fname in conditions:
    p = Path(fname)
    if p.exists():
        print(fmt_row(label, json.loads(p.read_text())['metrics']))
    else:
        print(f"{label:<30}  (not found — run cells above)")

print('\n=== Interpretation ===')
print('- Mode C >> Mode A (LoRA): collaborative soft tokens beat text even when trained')
print('- Mode C ≈ Mode A (LoRA): training was the main advantage, not modality')
print('- Mode A (LoRA) >> Mode A (zero-shot): confirms LoRA learned the task')

Condition                         HR@1    HR@3    HR@5   NDCG@1   NDCG@3   NDCG@5
---------------------------------------------------------------------------------
Random                           0.088   0.280   0.497   0.0877   0.1957   0.2842
Mode A (zero-shot)               0.121   0.376   0.545   0.1209   0.2637   0.3326
Mode A (LoRA)                    0.454   0.730   0.853   0.4536   0.6149   0.6653
Mode C (ranking adapt)           0.325   0.637   0.790   0.3245   0.5057   0.5681

=== Interpretation ===
- Mode C >> Mode A (LoRA): collaborative soft tokens beat text even when trained
- Mode C ≈ Mode A (LoRA): training was the main advantage, not modality
- Mode A (LoRA) >> Mode A (zero-shot): confirms LoRA learned the task


## Cell 7: Train Ranking Adapter on LoRA Backbone (Mode D)
Re-trains the MLP adapter using the **LoRA-finetuned LLM** as the frozen backbone instead of
the base model. The LoRA shifts the LLM's hidden space, so soft tokens trained against the
base model no longer align — we need a new adapter that projects SASRec embeddings into the
LoRA model's space.

This gives us Mode D: collaborative soft tokens + a task-tuned LLM backbone.

**Training time:** ~25-40 min on A100 for 8 epochs / 4000 examples.

In [11]:
LORA_MODEL = './llama31-1b-movielens-ranking-lora'

# Train a new ranking adapter with the LoRA model as the frozen backbone.
# Saves to checkpoints/adapter_ranking_lora.pt (separate from the base adapter).
!python scripts/train_adapter_ranking.py \
    --model {LORA_MODEL} \
    --epochs 10 --max-train 4000 --recon-lambda 0.3 --val-fraction 0.1 \
    --output checkpoints/adapter_ranking_lora.pt

[device] cuda
[config] model=./llama31-1b-movielens-ranking-lora  recon_lambda=0.3  chat_template=False
[data] 3509 movie titles loaded for reconstruction grounding
[data] 3600 train  /  400 val ranking examples
[load] PEFT adapter from llama31-1b-movielens-ranking-lora
[load] base model unsloth/Llama-3.2-1B-Instruct (download on first run ~2–5 min)
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 146/146 [00:00<00:00, 6143.53it/s]
[load] LLM ready (PEFT)
[load] merging LoRA into base model for inference...
[load] merge complete
[load] projected_embeddings cache loaded (3510, 2048)
[adapter] warm-start from: /

## Cell 8: Cache Projected Embeddings for LoRA Adapter
Projects all SASRec embeddings through the new adapter (using the LoRA model) and saves to
`checkpoints/projected_embeddings_ranking_lora.pt`.

In [14]:
!python scripts/cache_projected_embeddings.py \
    --adapter checkpoints/adapter_ranking_lora.pt \
    --out checkpoints/projected_embeddings_ranking_lora.pt

[data] 3510 items, sasrec_dim=50
[adapter] adapter_ranking_lora.pt  (contrastive_ranking)
[adapter] 50 → 1024 → 2048
[adapter] trained against: ./llama31-1b-movielens-ranking-lora
[adapter] best_val_HR@1=0.4125
[done] saved checkpoints/projected_embeddings_ranking_lora.pt  shape=(3510, 2048)


## Cell 9: Evaluate Mode D (LoRA backbone + soft tokens)
Runs Mode C eval but with the LoRA model and the new adapter.

In [17]:
LORA_MODEL = './llama31-1b-movielens-ranking-lora'


!python scripts/eval_ranking.py --modes C \
    --model {LORA_MODEL} \
    --embedding-adapter checkpoints/adapter_ranking_lora.pt \
    --projected-embeddings checkpoints/projected_embeddings_ranking_lora.pt \
    --n-candidates 10

!cp results_mode_C.json results_mode_C_lora.json
print('\n[saved] Mode D (LoRA + soft tokens) results -> results_mode_C_lora.json')

[device] cuda
[data] 604 test examples
[setup] modes=['C'] candidates=10 chat_template=False
[llm] llama31-1b-movielens-ranking-lora
[load] loading LLM (first run may download base weights)...
[adapter] adapter_ranking_lora.pt  method=contrastive_ranking
[adapter] trained against: ./llama31-1b-movielens-ranking-lora
[adapter] best_val_HR@1=0.4125
[cache] projected_embeddings: projected_embeddings_ranking_lora.pt
[load] PEFT adapter from llama31-1b-movielens-ranking-lora
[load] base model unsloth/Llama-3.2-1B-Instruct (download on first run ~2–5 min)
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Loading weights: 100% 146/146 [00:

## Cell 10: Final 5-Condition Comparison Table
All conditions: Random, Mode A zero-shot, Mode C (base), Mode A LoRA, Mode D (LoRA + soft tokens).

In [18]:
import json
from pathlib import Path

conditions = [
    ('Mode A (zero-shot)',          'results_mode_A_zeroshot.json'),
    ('Mode C (base adapter)',       'results_mode_C_base.json'),
    ('Mode A (LoRA)',               'results_mode_A_lora.json'),
    ('Mode D (LoRA + soft tokens)', 'results_mode_C_lora.json'),
]

ks, rand = [], {}
for _, f in conditions:
    p = Path(f)
    if p.exists():
        data = json.loads(p.read_text())
        ks   = sorted(int(k.split('@')[1]) for k in data['metrics'] if k.startswith('HR@'))
        rand = data.get('random_baseline', {})
        break

def fmt_row(label, m):
    hr   = ''.join(f"{m.get(f'HR@{k}',   0):>8.3f}" for k in ks)
    ndcg = ''.join(f"{m.get(f'NDCG@{k}', 0):>9.4f}" for k in ks)
    return f"{label:<32}{hr}{ndcg}"

hdr = f"{'Condition':<32}" + ''.join(f"{'HR@'+str(k):>8}" for k in ks) + ''.join(f"{'NDCG@'+str(k):>9}" for k in ks)
print(hdr)
print('-' * len(hdr))
if rand:
    print(fmt_row('Random', rand))
for label, fname in conditions:
    p = Path(fname)
    if p.exists():
        print(fmt_row(label, json.loads(p.read_text())['metrics']))
    else:
        print(f"{label:<32}  (not found — run cells above)")

print('\n=== 2x2 Design ===')
print('                     Base LLM        LoRA LLM')
print('Text candidates:     Mode A z-shot   Mode A LoRA')
print('Soft-token cands:    Mode C          Mode D')
print()
print('=== Interpretation ===')
print('- Mode D > Mode A LoRA: soft tokens add signal on top of a tuned LLM (additive)')
print('- Mode D ≈ Mode A LoRA: LoRA already captured all useful signal; soft tokens redundant')
print('- Mode D < Mode A LoRA: adapter/LoRA embeddings interfere (misaligned spaces)')

Condition                           HR@1    HR@3    HR@5   NDCG@1   NDCG@3   NDCG@5
-----------------------------------------------------------------------------------
Random                             0.088   0.280   0.497   0.0877   0.1957   0.2842
Mode A (zero-shot)                 0.121   0.376   0.545   0.1209   0.2637   0.3326
Mode C (base adapter)              0.325   0.637   0.790   0.3245   0.5057   0.5681
Mode A (LoRA)                      0.454   0.730   0.853   0.4536   0.6149   0.6653
Mode D (LoRA + soft tokens)        0.290   0.634   0.810   0.2897   0.4877   0.5603

=== 2x2 Design ===
                     Base LLM        LoRA LLM
Text candidates:     Mode A z-shot   Mode A LoRA
Soft-token cands:    Mode C          Mode D

=== Interpretation ===
- Mode D > Mode A LoRA: soft tokens add signal on top of a tuned LLM (additive)
- Mode D ≈ Mode A LoRA: LoRA already captured all useful signal; soft tokens redundant
- Mode D < Mode A LoRA: adapter/LoRA embeddings interfere (mis

In [19]:
import json
from pathlib import Path
from collections import defaultdict

LETTERS = 'ABCDEFGHIJ'

result_files = {
    'Mode A (zero-shot)':      'results_mode_A_zeroshot.json',
    'Mode C (base adapter)':   'results_mode_C_base.json',
    'Mode A (LoRA)':           'results_mode_A_lora.json',
    'Mode D (LoRA+soft)':      'results_mode_C_lora.json',
}

def analyze_letters(preds):
    n = len(preds)
    avg_prob      = defaultdict(float)
    pred_freq     = defaultdict(int)
    true_freq     = defaultdict(int)
    correct_by    = defaultdict(lambda: [0, 0])  # [correct, total]

    for p in preds:
        for letter, prob in p['probs'].items():
            avg_prob[letter] += prob
        pred_freq[p['pred_letter']] += 1
        true_freq[p['true_letter']] += 1
        correct_by[p['true_letter']][1] += 1
        if p['correct']:
            correct_by[p['true_letter']][0] += 1

    rows = []
    for letter in LETTERS:
        ap   = avg_prob[letter] / n
        pf   = 100 * pred_freq[letter] / n
        tf   = 100 * true_freq[letter] / n
        c, t = correct_by[letter]
        hr1c = 100 * c / t if t > 0 else 0.0
        rows.append((letter, ap, pf, tf, hr1c))

    top_letter = max(pred_freq, key=pred_freq.get)
    top_pct    = 100 * pred_freq[top_letter] / n
    mean_conf  = sum(p['probs'][p['pred_letter']] for p in preds) / n
    overall_hr1 = sum(p['correct'] for p in preds) / n
    return rows, top_letter, top_pct, mean_conf, overall_hr1

for label, fname in result_files.items():
    p = Path(fname)
    if not p.exists():
        print(f"\n[SKIP] {label}: {fname} not found")
        continue

    preds = json.loads(p.read_text())['predictions']
    rows, top_letter, top_pct, mean_conf, overall_hr1 = analyze_letters(preds)

    print(f"\n{'='*62}")
    print(f"{label}  (n={len(preds)})")
    print(f"{'='*62}")
    print(f"{'Letter':>8} {'AvgP':>7} {'PredFreq%':>10} {'TrueFreq%':>10} {'HR@1|pos%':>10}")
    print(f"{'-'*8:>8} {'-'*7:>7} {'-'*10:>10} {'-'*10:>10} {'-'*10:>10}")
    for letter, ap, pf, tf, hr1c in rows:
        print(f"{letter:>8} {ap:>7.4f} {pf:>10.1f} {tf:>10.1f} {hr1c:>10.1f}")
    print(f"\n  Top prediction: '{top_letter}' chosen {top_pct:.1f}% of the time")
    print(f"  Mean P(top-1):  {mean_conf:.4f}")
    print(f"  Overall HR@1:   {overall_hr1:.4f}")

# ── Side-by-side prediction concentration summary ────────────────────────
print(f"\n\n{'='*62}")
print("POSITION BIAS SUMMARY")
print(f"{'='*62}")
print(f"{'Condition':<26} {'Top letter':>10} {'Top freq%':>10} {'Mean conf':>10} {'HR@1':>8}")
print('-' * 66)
for label, fname in result_files.items():
    p = Path(fname)
    if not p.exists():
        continue
    preds = json.loads(p.read_text())['predictions']
    _, top_letter, top_pct, mean_conf, overall_hr1 = analyze_letters(preds)
    print(f"{label:<26} {top_letter:>10} {top_pct:>10.1f} {mean_conf:>10.4f} {overall_hr1:>8.4f}")


Mode A (zero-shot)  (n=604)
  Letter    AvgP  PredFreq%  TrueFreq%  HR@1|pos%
-------- ------- ---------- ---------- ----------
       A  0.1924       66.4       11.1       73.1
       B  0.0887        0.5        7.9        0.0
       C  0.1101        2.5       10.9        4.5
       D  0.0647        0.0       10.1        0.0
       E  0.0831        0.3        8.8        0.0
       F  0.0673        0.2       11.1        0.0
       G  0.1067        2.6        9.6        5.2
       H  0.0444        0.0       10.9        0.0
       I  0.0724        0.7        9.1        1.8
       J  0.1701       26.8       10.4       27.0

  Top prediction: 'A' chosen 66.4% of the time
  Mean P(top-1):  0.2076
  Overall HR@1:   0.1209

Mode C (base adapter)  (n=604)
  Letter    AvgP  PredFreq%  TrueFreq%  HR@1|pos%
-------- ------- ---------- ---------- ----------
       A  0.0825        6.3       11.1       23.9
       B  0.0935       10.1        7.9       35.4
       C  0.0742        7.1       10.9   